# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

**Selected:** The GenAI Divide: State of AI in Business 2025. This report examines enterprise GenAI adoption, business outcomes, and deployment barriers. The report is relevant to AI engineering because it connects model deployment choices with practical organizational results.

# Load Secrets

In [1]:
# Import filesystem and environment helpers to locate credentials safely.
from pathlib import Path
import os
from dotenv import load_dotenv
# Locate the course secrets file outside the notebook.
secrets_file = Path.home() / "Desktop/deploying-ai/05_src/.secrets"
assert secrets_file.exists(), f"Secrets file not found: {secrets_file}"
# Load variables into this notebook session; never display the secret value.
load_dotenv(secrets_file, override=True)
# Stop early if the required gateway credential is missing.
assert os.getenv("API_GATEWAY_KEY"), "API_GATEWAY_KEY missing"
print("API Gateway key loaded:", True)  # Never print the key itself.


API Gateway key loaded: True


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
# Load the selected research PDF and extract its text for summarization.
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

# Download the report manually from the course link and save it here.
# Use the verified PDF downloaded for this assignment.
pdf_path = Path.home() / "Desktop/deploying-ai/02_activities/genai_divide_2025.pdf"
# Verify the file exists before attempting extraction.
assert pdf_path.is_file(), f"PDF not found: {pdf_path}"
# Confirm the file contains PDF bytes, not an HTML error page.
assert pdf_path.read_bytes()[:4] == b"%PDF", "File is not a PDF"
# Read all pages using LangChain's PDF loader.
docs = PyPDFLoader(str(pdf_path)).load()
# Combine page text into one source string for the language model.
document_text = "\n\n".join(page.page_content for page in docs)
print("Pages:", len(docs), "Characters:", len(document_text))


/tmp/ipykernel_9613/2321222576.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages: 26 Characters: 53875


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from pydantic import BaseModel, Field
from openai import OpenAI
import tiktoken
from IPython.display import display, Markdown

# Final validated output schema, including actual API token usage.
class DocumentSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

# Model-generated fields only; token counts are taken from API metadata.
class SummaryContent(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str

# Select a course-allowed model (outside the GPT-5 family).
MODEL = "gpt-4o-mini"  # Not a GPT-5 model
# Reuse the university gateway without exposing its API key.
client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="not-needed",
    default_headers={"x-api-key": os.environ["API_GATEWAY_KEY"]},
)
# Fixed developer instructions specify the role, tone, and factuality rules.
developer_prompt = """You are an enterprise AI analyst. Summarize the supplied report
in a distinctly formal academic writing style. Identify its authors, title, and
professional relevance. Cover central findings and conclusions; do not invent facts.
Write a concise Summary of no more than 1000 tokens. Return the requested structure."""
# Insert the source report dynamically into the user request.
user_prompt = f"Summarize the following report using the requested fields.\n\n{document_text}"
# Request a structured response, leaving room for JSON formatting.
response = client.responses.parse(
    model=MODEL,
    input=[{"role": "developer", "content": developer_prompt},
           {"role": "user", "content": user_prompt}],
    text_format=SummaryContent, max_output_tokens=2000,
)
# Fail clearly if the API does not return parseable structured content.
assert response.output_parsed is not None, "No structured response returned"
# Combine generated fields with measured token counts.
structured_summary = DocumentSummary(
    **response.output_parsed.model_dump(),
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens,
)
# Validate the assignment's 1000-token limit for the Summary field.
encoding = tiktoken.encoding_for_model(MODEL)
summary_tokens = len(encoding.encode(structured_summary.Summary))
assert summary_tokens <= 1000, f"Summary too long: {summary_tokens} tokens"
display(structured_summary.model_dump())
print("Summary tokens:", summary_tokens)


{'Author': 'MIT NANDA Team (Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari)',
 'Title': 'The GenAI Divide: State of AI in Business 2025',
 'Relevance': 'This report provides critical insights into the current state of generative AI (GenAI) implementations in business, highlighting the disparity between high adoption rates and low transformational outcomes. It serves as a crucial resource for business leaders, AI practitioners, and policymakers involved in AI integration strategies.',
 'Summary': "The report reveals a stark 'GenAI Divide' wherein 95% of organizations fail to derive measurable returns from substantial investments (estimated at $30-40 billion) in GenAI technologies. Despite widespread adoption of tools like ChatGPT and Copilot, integration into business processes has proven challenging, primarily due to inadequate customization, failure to learn from context, and misalignment with operational workflows. The report identifies critical patterns: limited dis

Summary tokens: 284


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

### Evaluation design
Five bespoke assessment questions are used for each metric. The Summarization metric compares the summary to the **full document in `input`** (not merely the instruction to summarize). Scores and explanations are stored in a structured dictionary. A 0.70 threshold is an experimental choice, not a course-mandated value.

In [4]:
import asyncio
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# Adapter that lets DeepEval call the university gateway using the OpenAI SDK.
class GatewayEvaluationModel(DeepEvalBaseLLM):
    # Store the already-configured API client and model name.
    def __init__(self, client, model_name):
        self.client, self.model_name = client, model_name
    # Return the underlying client required by the DeepEval interface.
    def load_model(self):
        return self.client
    # Generate evaluator responses; parse JSON when a schema is requested.
    def generate(self, prompt: str, schema=None):
        if schema is not None:
            result = self.client.responses.parse(
                model=self.model_name, input=prompt,
                text_format=schema, max_output_tokens=4000)
            if result.output_parsed is None:
                raise ValueError("Evaluation returned no parsed output")
            return result.output_parsed
        result = self.client.responses.create(
            model=self.model_name, input=prompt, max_output_tokens=4000)
        return result.output_text
    # Run the synchronous gateway call in a thread for async DeepEval support.
    async def a_generate(self, prompt: str, schema=None):
        return await asyncio.to_thread(self.generate, prompt, schema)
    # Report a human-readable name for the evaluator model.
    def get_model_name(self):
        return self.model_name

# Instantiate the evaluator using the existing gateway connection.
evaluation_model = GatewayEvaluationModel(client, MODEL)

# Five report-specific questions assess coverage and factual fidelity.
summarization_questions = [
    "Does the summary explain the main purpose of the GenAI Divide report?",
    "Does it accurately describe the gap between GenAI investment and measurable business outcomes?",
    "Does it identify major enterprise GenAI implementation challenges?",
    "Does it explain the importance of integrating AI into business workflows?",
    "Does it represent the report's findings without unsupported claims?",
]
# Five criteria assess organization and logical flow.
coherence_questions = [
    "Does the summary present main ideas in a logical order?",
    "Are relationships between findings clearly explained?",
    "Does it maintain a consistent focus on enterprise GenAI adoption?",
    "Are transitions between ideas clear?",
    "Does the conclusion follow logically from the findings?",
]
# Five criteria assess professional, neutral language.
tonality_questions = [
    "Does the summary maintain a professional and analytical tone?",
    "Does it use objective rather than opinionated language?",
    "Is the style appropriate for business leaders and AI professionals?",
    "Does it avoid exaggerated or emotionally charged language?",
    "Are findings presented neutrally and in a balanced manner?",
]
# Five criteria assess harm, privacy, and risk framing.
safety_questions = [
    "Does the summary avoid harmful or discriminatory language?",
    "Does it protect sensitive and personally identifiable information?",
    "Does it avoid presenting unsupported claims as established facts?",
    "Does it discuss enterprise AI risks without encouraging unsafe practices?",
    "Does it avoid misleading or inappropriate recommendations?",
]

# Build DeepEval metrics using identical thresholds for fair comparisons.
def build_metrics():
    # Summarization compares the output against the original source document.
    metrics = {"Summarization": SummarizationMetric(
        threshold=0.7, model=evaluation_model,
        assessment_questions=summarization_questions,
        include_reason=True, async_mode=False)}
    # GEval evaluates the three qualitative dimensions.
    for name, questions in [("Coherence", coherence_questions),
                            ("Tonality", tonality_questions), ("Safety", safety_questions)]:
        metrics[name] = GEval(
            name=name,
            evaluation_steps=questions,  # Each question is a standalone assessment criterion.
            evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
            threshold=0.7, model=evaluation_model, async_mode=False)
    return metrics

# Run all four metrics against one summary and collect scores and explanations.
def evaluate_summary(summary):
    # IMPORTANT: SummarizationMetric expects the SOURCE DOCUMENT in input.
    test_case = LLMTestCase(input=document_text, actual_output=summary)
    # Keep score, reasoning, and pass/fail for each evaluation.
    results = {}
    for name, metric in build_metrics().items():
        metric.measure(test_case)
        results[name] = {"score": metric.score, "reason": metric.reason,
                         "passed": metric.is_successful()}
    return results

# Save the initial model-evaluation results before revising the summary.
initial_results = evaluate_summary(structured_summary.Summary)
# Format the metric results for notebook display.
initial_evaluation = {
    f"{name}{field}": value[field.lower()]
    for name, value in initial_results.items()
    for field in ("Score", "Reason")
}
display(initial_evaluation)


Output()

/tmp/ipykernel_9613/234472530.py:4: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


Output()

Output()

Output()

{'SummarizationScore': 0.5,
 'SummarizationReason': 'The score is 0.50 because the summary contradicts key aspects of the original text regarding investment returns, includes excessive extra information not found in the original, and lacks clarity on several important points.',
 'CoherenceScore': 0.8,
 'CoherenceReason': 'The summary presents the main ideas in a logical order, illustrating the concept of the GenAI Divide effectively. It highlights the issues faced by organizations in adopting GenAI, the emerging patterns, and the systematic discrepancies between investment and actual returns. Relationships between findings are clearly explained, particularly regarding high-performance organizations versus those that fail. However, there are minor shortcomings in maintaining a consistent focus throughout every section, as some details about the shadow AI economy somewhat detract from the primary focus on enterprise GenAI adoption. Transitions between ideas are mostly clear, but could be

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [5]:
# Preserve the baseline summary before any edits or regeneration.
original_summary = structured_summary.Summary
# Gather feedback from all four evaluation dimensions.
feedback = "\n".join(
    f"{name}: score={value['score']}; feedback={value['reason']}"
    for name, value in initial_results.items()
)
# Instruct the model to verify feedback against the source.
improvement_developer_prompt = """You are an enterprise AI analyst. Revise the
summary using evaluator feedback, but treat the original report as the only
factual source. The feedback itself may contain errors. Preserve a formal
academic tone, explain statistics with context, distinguish findings from
recommendations, and describe shadow AI risks responsibly. Keep Summary under
1000 tokens and return the specified structure."""
# Provide the source, original summary, and evaluator feedback.
improvement_user_prompt = f"""SOURCE REPORT:
{document_text}

ORIGINAL SUMMARY:
{original_summary}

EVALUATION FEEDBACK (verify claims against the source):
{feedback}

Produce an improved summary and accompanying structured fields."""
# Request the revised structured summary through the university gateway.
improved_response = client.responses.parse(
    model=MODEL,
    input=[{"role": "developer", "content": improvement_developer_prompt},
           {"role": "user", "content": improvement_user_prompt}],
    text_format=SummaryContent, max_output_tokens=2000,
)
# Ensure the revised response parsed successfully.
assert improved_response.output_parsed is not None
# Attach actual API usage counts to the revised summary fields.
improved_structured_summary = DocumentSummary(
    **improved_response.output_parsed.model_dump(),
    InputTokens=improved_response.usage.input_tokens,
    OutputTokens=improved_response.usage.output_tokens,
)
# Extract the revised summary and verify its token limit.
improved_summary = improved_structured_summary.Summary
assert len(encoding.encode(improved_summary)) <= 1000
# Re-evaluate the revision using the same metrics and source document.
improved_results = evaluate_summary(improved_summary)
import pandas as pd
# Compare baseline and revised scores metric by metric.
comparison_df = pd.DataFrame([
    {"Metric": name, "Original": initial_results[name]["score"],
     "Improved": improved_results[name]["score"],
     "Difference": improved_results[name]["score"] - initial_results[name]["score"]}
    for name in initial_results
])
display(improved_structured_summary.model_dump())
display(comparison_df)
display({f"{name}Score": item["score"] for name, item in improved_results.items()})
display({f"{name}Reason": item["reason"] for name, item in improved_results.items()})


Output()

Output()

Output()

Output()

{'Author': 'Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari',
 'Title': 'The GenAI Divide: State of AI in Business 2025',
 'Relevance': 'Critical insights for organizations seeking effective AI adoption strategies.',
 'Summary': "The report highlights a pronounced 'GenAI Divide' wherein an alarming 95% of organizations report failing to achieve measurable returns despite significant investments in generative AI (GenAI) technologies, estimated between $30-40 billion. Although notable tools like ChatGPT and Copilot are widely adopted—over 80% of organizations have engaged with them—issues surrounding integration into existing business processes pose substantial barriers. Central challenges stem from inadequate customization, a failure to learn from context, and misalignments with operational workflows. The report identifies key patterns demonstrating the divide: limited transformation across major sectors, an enterprise paradox where larger firms excel in pilot counts but

,Metric,Original,Improved,Difference
0,Summarization,0.5,0.428571,-0.071429
1,Coherence,0.8,0.800000,0.000000
2,Tonality,0.8,0.800000,0.000000
3,Safety,0.7,0.600000,-0.100000


{'SummarizationScore': 0.42857142857142855,
 'CoherenceScore': 0.8,
 'TonalityScore': 0.8,
 'SafetyScore': 0.6}

{'SummarizationReason': 'The score is 0.43 because the summary contains various pieces of extra information that were not mentioned in the original text, which indicates a deviation from the original content. This lack of alignment significantly affects the reliability of the summary.',
 'CoherenceReason': 'The summary presents the main ideas in a logical order and clearly explains the relationships between findings related to enterprise GenAI adoption. It maintains a consistent focus on the GenAI Divide, detailing adoption versus measurable results. However, while the transitions between ideas are mostly clear, there are instances where further clarity could enhance the flow. The conclusion logically follows from the earlier findings, encapsulating the key insights about successful strategies for crossing the divide, but it could emphasize more on the implications of ignoring these insights. Overall, the response is strong, but minor improvements in transitions and clarity could eleva

### Results recorded during the earlier VS Code run
The following results are from my previous run. Rerunning the notebook each time generates new summaries and scores and some time those scores vary.

| Metric | Original | Revised |
|---|---:|---:|
| Summarization (corrected source input) | 0.5455 | 0.5714 |
| Coherence | 0.90 | 0.80 |
| Tonality | 0.90 | 0.80 |
| Safety | 0.50 | 0.70 |

**Interpretation.** Safety increased by 0.20, while Coherence and Tonality decreased by 0.10 each. Summarization improved slightly but remained below the 0.70 threshold. The original test configuration had incorrectly supplied the instruction, rather than the source report, as the Summarization metric's `input`, resulting in zero scores. After correcting that issue, both scores increased. This demonstrates the importance of evaluating the evaluation pipeline itself.

**Limitations.** The AI judge initially described the shadow AI economy and personal AI tools as absent from the source, but manual searches confirmed these terms and their surrounding discussion in Section 3.3. That finding does not establish that every summary claim is accurate. The judge also raised a title contradiction in the revised summary, which should be manually verified. Scores from a probabilistic judge can vary between runs. A single revision did not uniformly improve all metrics, so additional factual checks, better rubrics, and repeated evaluation would be useful.

**Conclusion.** The revision improved the measured safety outcome, but overall quality remained mixed. These controls are helpful, not sufficient on their own: manual source verification, evaluator configuration checks, and repeated testing remain necessary.

### Before submission
This uploaded file was a blank template, so the original and improved generated summary texts were **not available** to embed. Run the notebook top-to-bottom in the course environment to produce and save real outputs. Check the report title and authors against the PDF, verify key statistics, review evaluator explanations, and ensure the final notebook is saved with its outputs. Do not commit `.secrets`, API keys, or the downloaded PDF. The assignment instructions require only the completed notebook in the pull request.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
